In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway_2019 = os.path.join(pathway_temp, "bohn2019natural")
original_data_pathway = os.path.join(pathway_2019, "original_data")
pathway_2020 = os.path.join(pathway_temp, "bohn2020learning")

complete_path_1 = os.path.join(original_data_pathway, "Bohn_2019_2020_part1_NaturalReference_Data.csv")
complete_path_2 = os.path.join(original_data_pathway, "Bohn_2019_2020_part2_NaturalReference_Data.csv")

out_2019_pathway = os.path.join(pathway_2019, "standardized_data")
if not os.path.exists(out_2019_pathway):
    os.makedirs(out_2019_pathway)

out_2020_pathway = os.path.join(pathway_2020, "standardized_data")
if not os.path.exists(out_2020_pathway):
    os.makedirs(out_2020_pathway)

In [2]:
import pandas as pd
import numpy as np
df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)

experiment_import = [[df1, 'p1', 'bohn2019natural'], 
                    [df2, 'p2', 'bohn2020learning']]
for x, y, k in experiment_import:                    
    x['experiment_name']=y
    x['study_id']=k

In [3]:
data_frames=[df1, df2]

for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"subject": "ape"})
    # x['study_id']="bohn2019natural"
    data_frames[index]=x
new_df=data_frames[0]

In [4]:
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

fulldf=fulldf.rename(columns={"group": "group_original"})

In [5]:
fulldf['ape'] = fulldf['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
# fulldf.columns

In [6]:
fulldf = fulldf.rename(columns={"species_y": "species"})
fulldf.rename(columns={"ape": "participant"}, inplace=True)

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

In [7]:
fulldf=fulldf[['study_id',   'year', 'month', 'day', 'participant', 'age_in_years',
         'sex', 'species', 'group_original', 'session', 'sessiontotal',
       'sessioncond', 'trial', 'condition',
       'appleft', 'appright', 'toneleft', 'toneright', 'indicated', 'choice', 'code', 'train',
         'apparatus',
        'solve', 'time', 'man', 'manleft', 'manright', 'biman']]

In [8]:
exp1 = fulldf[fulldf['study_id'] == 'bohn2019natural']
exp2 = fulldf[fulldf['study_id'] == 'bohn2020learning']
##request to remove obcho trials
exp1 = exp1[~exp1.condition.str.contains("obcho")]

experiments = [[exp1, 'bohn2019natural_exp1', out_2019_pathway], ##connects df with name of output dataset
                [ exp2, 'bohn2020learning_exp2', out_2020_pathway]]

for x,y, k in experiments:
    x = x.dropna(axis=1, how='all')## drop empty rows/columns
    comp_out_path_stand = os.path.join(k, y+'_standardized.csv')
    x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
    ##glossaries
    names = x.columns.tolist()
    df = pd.DataFrame(names)
    df = df.rename(columns={0: "column_name"})
    df["description"] = ""
    studyID_glossary=df[["column_name", "description"]]

    comp_out_path_glossary = os.path.join(k, y+'_glossary.csv')
    studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)